In [10]:
import keras
import numpy as np
from keras import layers
from keras.utils import to_categorical
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os

# 限制 GPU 記憶體使用
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            tf.config.experimental.set_virtual_device_configuration(
                gpu,
                [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)]  # 限制為 4GB
            )
    except RuntimeError as e:
        print(e)


In [11]:
num_classes = 10
input_shape = (28, 28, 1)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)


In [12]:
model = keras.Sequential(
    [
        layers.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
print("開始訓練分類模型")
model.fit(x_train, y_train, batch_size=128, epochs=3, validation_split=0.1)
print("分類模型訓練完成")

_ = model.predict(x_train[:1])


開始訓練分類模型
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9196 - loss: 0.2637 - val_accuracy: 0.9842 - val_loss: 0.0578
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9749 - loss: 0.0790 - val_accuracy: 0.9837 - val_loss: 0.0550
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9816 - loss: 0.0593 - val_accuracy: 0.9900 - val_loss: 0.0340
分類模型訓練完成
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step


In [13]:
def generate_fgsm_samples_batch(model, x_data, y_data, epsilon=0.1, batch_size=128):
    x_adv = []
    for i in range(0, len(x_data), batch_size):
        x_batch = x_data[i:i + batch_size]
        y_batch = y_data[i:i + batch_size]

        x_tensor = tf.convert_to_tensor(x_batch, dtype=tf.float32)
        y_true = tf.convert_to_tensor(y_batch, dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(x_tensor)
            predictions = model(x_tensor)
            loss = tf.keras.losses.categorical_crossentropy(y_true, predictions)

        gradients = tape.gradient(loss, x_tensor)
        x_batch_adv = x_batch + epsilon * tf.sign(gradients)
        x_batch_adv = tf.clip_by_value(x_batch_adv, 0, 1)
        x_adv.append(x_batch_adv.numpy())

    return np.vstack(x_adv)

print("開始生成 FGSM 攻擊樣本")
x_train_adv = generate_fgsm_samples_batch(model, x_train, y_train, epsilon=0.1)
print("FGSM 攻擊樣本生成完成")


開始生成 FGSM 攻擊樣本
FGSM 攻擊樣本生成完成


In [14]:
logit_layer_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[-2].output)

def extract_logits(model, data):
    return model.predict(data)

print("開始提取 Logit 層輸出")
logits_original = extract_logits(logit_layer_model, x_train[:100])
print(f'logits_original shape:{logits_original.shape}')
print(logits_original[0])
logits_adv = extract_logits(logit_layer_model, x_train_adv[:100])
print("Logit 層輸出提取完成")


開始提取 Logit 層輸出
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
logits_original shape:(100, 128)
[2.6583855  0.         0.89162636 2.3553402  0.         0.8801754
 3.350323   0.         4.116711   1.8015257  1.6939738  0.
 1.186212   0.         0.         1.2710406  0.         0.
 1.3971874  0.6041652  0.         0.         0.         0.
 0.         1.8497235  3.6590672  4.3847127  1.3980819  0.
 2.9294326  0.         4.072258   2.065241   2.1137614  3.2070293
 3.0414152  0.06320799 1.9022045  0.         0.         0.
 3.9392228  1.3869866  2.0597754  0.92152184 3.732782   0.
 3.777499   0.         1.5097525  0.         1.283145   0.21939881
 0.         3.920356   0.         0.48761958 0.         2.0137916
 2.4375734  0.         2.576372   0.         0.         0.
 0.17241612 0.         0.         0.         0.         1.9333962
 0.         0.         2.3844278  0.         0.         0.
 6.15696    0.         0.96983916 0.         3.390927   0.
 0.         0.40730265 0.         4.1722274  0.     

In [15]:
X = np.vstack([logits_original, logits_adv])
y = np.array([0] * len(logits_original) + [1] * len(logits_adv))  # 0: 原始樣本, 1: 攻擊樣本

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

detector_model = keras.Sequential(
    [
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(2, activation="softmax"),
    ]
)

detector_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
print("開始訓練檢測器模型")
detector_model.fit(X_train, y_train, batch_size=32, epochs=10, validation_split=0.1)
print("檢測器模型訓練完成")


開始訓練檢測器模型
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - accuracy: 0.4722 - loss: 0.8160 - val_accuracy: 0.5625 - val_loss: 0.6983
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5069 - loss: 0.7220 - val_accuracy: 0.4375 - val_loss: 0.7201
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5972 - loss: 0.6527 - val_accuracy: 0.5625 - val_loss: 0.7095
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6181 - loss: 0.6280 - val_accuracy: 0.4375 - val_loss: 0.6922
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6528 - loss: 0.5989 - val_accuracy: 0.5000 - val_loss: 0.6951
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7708 - loss: 0.5614 - val_accuracy: 0.4375 - val_loss: 0.7181
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7361 - loss: 0.5338 - val_accuracy: 0.5000 - val_loss: 0.6980
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7431 - loss: 0.5227 - val_accuracy: 0.5625 - val_lo

In [16]:
print("開始評估檢測器模型")
y_pred = np.argmax(detector_model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))


開始評估檢測器模型
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
              precision    recall  f1-score   support

           0       0.40      0.57      0.47        21
           1       0.10      0.05      0.07        19

    accuracy                           0.33        40
   macro avg       0.25      0.31      0.27        40
weighted avg       0.26      0.33      0.28        40

